# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

In [1]:
#@title ▶️ Run OmniVoice (GPU auto-stops when idle)
#@markdown Run ONLY this cell. It installs OmniVoice, loads the model, opens the web UI,
#@markdown and **shuts the GPU runtime down automatically** after it sits unused.
IDLE_MINUTES = 10  #@param {type:"integer"}
MAX_SESSION_HOURS = 2  #@param {type:"number"}
AUTO_TRANSCRIBE_REF_AUDIO = True  #@param {type:"boolean"}

import os, sys, time, threading, subprocess

# ---------- safety net FIRST: even a failed install/load can't burn credits ----------
from google.colab import runtime as _colab_runtime
_ov = {"start": time.time(), "last": time.time(), "busy": 0, "stopped": False}
_ov_lock = threading.Lock()

def stop_gpu(reason="manual"):
    """Release the Colab VM so it stops using compute units."""
    if _ov["stopped"]:
        return
    _ov["stopped"] = True
    print(f"\n⏹ Stopping GPU runtime ({reason}) to save credits...", flush=True)
    try:
        _colab_runtime.unassign()
    except Exception as e:
        print("unassign() failed:", e, "- killing kernel instead", flush=True)
        os.kill(os.getpid(), 9)

def _watchdog():
    while not _ov["stopped"]:
        time.sleep(20)
        now = time.time()
        with _ov_lock:
            busy, idle = _ov["busy"], now - _ov["last"]
        if busy == 0 and idle > IDLE_MINUTES * 60:
            stop_gpu(f"idle for {IDLE_MINUTES} min")
        elif now - _ov["start"] > MAX_SESSION_HOURS * 3600:
            stop_gpu(f"hit {MAX_SESSION_HOURS} h session cap")

if not any(t.name == "ov_watchdog" for t in threading.enumerate()):
    threading.Thread(target=_watchdog, name="ov_watchdog", daemon=True).start()

def _tracked(fn):
    def wrapper(*a, **k):
        with _ov_lock:
            _ov["busy"] += 1
        try:
            return fn(*a, **k)
        finally:
            with _ov_lock:
                _ov["busy"] -= 1
                _ov["last"] = time.time()
    return wrapper

# ---------- install ----------
try:
    import omnivoice  # already installed in this runtime
except ImportError:
    print("Installing OmniVoice (~1-2 min)...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice"], check=True)

# ---------- load model ----------
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
from omnivoice import OmniVoice
from omnivoice.cli.demo import build_demo

if "model" not in globals():
    print("Loading model...", flush=True)
    model = OmniVoice.from_pretrained(
        "k2-fsa/OmniVoice",
        device_map="cuda:0",
        dtype=torch.float16,
        load_asr=AUTO_TRANSCRIBE_REF_AUDIO,
    )
    # every generation (web UI or Python) counts as activity
    model.generate = _tracked(model.generate)
    model.create_voice_clone_prompt = _tracked(model.create_voice_clone_prompt)
_ov["last"] = time.time()

# ---------- web UI ----------
demo = build_demo(model, "k2-fsa/OmniVoice")
demo.queue().launch(share=True, debug=False)
print(f"\n✅ Ready. Open the public link above. GPU auto-stops after {IDLE_MINUTES} min "
      f"with no generations (hard cap {MAX_SESSION_HOURS} h). Run stop_gpu() to stop now.")


Installing OmniVoice (~1-2 min)...


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Loading model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/omnivoice/cli/demo.py:296: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, title="OmniVoice Demo") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://475bd12295adebf6bd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



✅ Ready. Open the public link above. GPU auto-stops after 10 min with no generations (hard cap 2 h). Run stop_gpu() to stop now.


In [2]:
#@title 🎙️ Andrew-style US male narrator — make samples
import torch, soundfile as sf
from IPython.display import Audio, display

SAMPLE_TEXT = ("The rain had not stopped for three days. Lin Feng stood at the edge of the cliff, "
               "staring into the mist, and for the first time in years, he smiled.")
STYLES = {
    "A": "male, young adult, low pitch, american accent",
    "B": "male, young adult, moderate pitch, american accent",
    "C": "male, middle-aged, low pitch, american accent",
}
for key, instruct in STYLES.items():
    for seed in (1, 2):
        torch.manual_seed(seed)
        audio = model.generate(text=SAMPLE_TEXT, instruct=instruct)
        name = f"andrew_{key}{seed}.wav"
        sf.write(name, audio[0], 24000)
        print(f"{key}{seed}  ({instruct})")
        display(Audio(audio[0], rate=24000))

# After you pick one (e.g. "B1"), lock it in so every chapter uses the SAME voice:
def use_voice(pick):
    global andrew
    andrew = model.create_voice_clone_prompt(ref_audio=f"andrew_{pick}.wav", ref_text=SAMPLE_TEXT)
    andrew.save("andrew_voice.pt")
    print(f"Locked voice {pick} -> andrew_voice.pt")

def narrate(text, out="narration.wav", speed=1.0):
    audio = model.generate(text=text, voice_clone_prompt=andrew, speed=speed)
    sf.write(out, audio[0], 24000)
    display(Audio(audio[0], rate=24000))
    return out


A1  (male, young adult, low pitch, american accent)


A2  (male, young adult, low pitch, american accent)


B1  (male, young adult, moderate pitch, american accent)


B2  (male, young adult, moderate pitch, american accent)


C1  (male, middle-aged, low pitch, american accent)


C2  (male, middle-aged, low pitch, american accent)


## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [8]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [omnivoice]


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [9]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [5]:
import glob, os
candidates = []
for pattern in ("*.wav", "*.mp3", "*.mpeg", "*.flac", "*.m4a", "*.ogg"):
    candidates += glob.glob(pattern)
print("Audio files:", candidates)
ref_audio_path = next((p for p in candidates if "andrew" in p.lower()), candidates[-1])
print("Using reference:", ref_audio_path)

Audio files: ['andrew voice .mpeg']
Using reference: andrew voice .mpeg


In [10]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    ref_audio=ref_audio_path,
    # ref_text="Transcription of the reference audio.",  # optional
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

In [11]:
# Andrew voice profile — reuse for every episode
andrew = model.create_voice_clone_prompt(ref_audio=ref_audio_path)
andrew.save("andrew_voice.pt")

def narrate(text, out="andrew_narration.wav", speed=1.0):
    audio = model.generate(text=text, voice_clone_prompt=andrew, speed=speed)
    sf.write(out, audio[0], 24000)
    display(Audio(audio[0], rate=24000))
    return out

print("Saved reusable Andrew voice profile: andrew_voice.pt")

Saved reusable Andrew voice profile: andrew_voice.pt


In [20]:
import numpy as np, librosa
text_15s = ("For three days, the rain had hidden the mountains beneath a silver veil. "
"At the cliff's edge, Lin Feng listened to the distant thunder, drew a slow breath, "
"and stepped forward, knowing the journey ahead would change everything.")
audio_raw = model.generate(text=text_15s, voice_clone_prompt=andrew, speed=0.8)[0].astype(np.float32)
target_samples = 15 * 24000
rate = len(audio_raw) / target_samples
audio_exact = librosa.effects.time_stretch(audio_raw, rate=rate)
audio_exact = np.pad(audio_exact, (0, max(0, target_samples - len(audio_exact))))[:target_samples]
sf.write("andrew_15_second_sample.wav", audio_exact, 24000)
display(Audio(audio_exact, rate=24000))
print("Generated exact 15-second file: andrew_15_second_sample.wav")

Generated exact 15-second file: andrew_15_second_sample.wav


In [19]:
import soundfile as sf
info_15s = sf.info("andrew_15_second_sample.wav")
print(f"Duration: {info_15s.duration:.2f} seconds")

Duration: 13.61 seconds


### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))